# Routed ZZZX Lattice Surgery

-- `UnrotatedRoutedMultiPatchCoupler`

This notebook follows the basis-aware routed surgery design:

- First infer which logical basis is actually attached to the ancillary patch region.
- Apply logical H only when the target Pauli differs from that interface basis.
- For true X/Z mixed interfaces, generate local mixed templates at seams/corners instead of assuming every stabilizer has the same type or weight.

The first experiment is a DEM-validated ZZZX readout using a Z-interface normalization. The second experiment shows the native mixed-interface geometry and local weight-2/3/4 templates.

In [ ]:
import sys, os, importlib
sys.path.insert(0, os.path.abspath('../..'))

# Reload local modules so a long-lived VS Code kernel does not keep stale classes.
import lightstim.qec_code.surface_code.unrotated.multi_patch_coupler as _multi_patch_coupler
import lightstim.qec_code.surface_code.unrotated.SE_block as _se_block
import lightstim.protocols.routed_multi_patch_ls as _routed_ls
importlib.reload(_multi_patch_coupler)
importlib.reload(_se_block)
importlib.reload(_routed_ls)

from lightstim.qec_code.surface_code.unrotated.code_patch import UnrotatedSurfaceCode
from lightstim.qec_code.surface_code.unrotated.SE_block import UnrotatedSurfaceCodeExtractionBlock
from lightstim.qec_code.surface_code.unrotated.multi_patch_coupler import UnrotatedRoutedMultiPatchCoupler
from lightstim.ir.qec_system import QECSystem
from lightstim.ir.builder import CircuitBuilder
from lightstim.ir.tracker import SyndromeTracker
from lightstim.protocols.routed_multi_patch_ls import (
    basis_change_indices_for_interfaces,
    build_routed_pauli_product_readout_circuit,
    infer_interface_paulis,
    routed_coupler_data_basis,
    solve_routed_pauli_product_syndromes,
)


def make_system(d=3):
    layout = [
        ('p1', (0, 0)),
        ('p2', (20, 0)),
        ('p3', (10, 20)),
        ('p4', (30, 20)),
    ]
    system = QECSystem()
    for name, off in layout:
        system.add_patch(UnrotatedSurfaceCode(distance=d), name=name, offset=off)
    return system


patch_names = ['p1', 'p2', 'p3', 'p4']
target_paulis = 'ZZZX'


## Basis rule

The H decision is made by comparing the requested Pauli with the basis of the side that actually touches the ancillary patch region.

In [ ]:
system = make_system()
for side in ['left', 'right', 'top', 'bottom']:
    basis = UnrotatedRoutedMultiPatchCoupler.infer_side_basis(system.patches['p1'][0], side)
    print(f'p1:{side:>6} -> {basis}')

print('XZ target on X/Z interfaces needs H at:',
      basis_change_indices_for_interfaces('XZ', ['X', 'Z']))
print('ZZZX target on Z/Z/Z/Z interfaces needs H at:',
      basis_change_indices_for_interfaces('ZZZX', ['Z', 'Z', 'Z', 'Z']))
print('ZZZX target on Z/Z/Z/X interfaces needs H at:',
      basis_change_indices_for_interfaces('ZZZX', ['Z', 'Z', 'Z', 'X']))


---

## Exp 1: legacy DEM check for ZZZX via Z-interface normalization

This cell keeps the existing detector-error-model-validated final-readout path. It intentionally uses the legacy thin route (`route_width=1`) and therefore does **not** display `circuit.diagram('detslice-with-ops-svg')`. The full ancillary-patch geometry and `detslice` diagram are shown in Exp 2.

In [ ]:
d = 3
system = make_system(d)
z_normalized_sides = ['right', 'left', 'right', 'left']
z_interfaces = ['Z', 'Z', 'Z', 'Z']

circuit, info, system = build_routed_pauli_product_readout_circuit(
    system=system,
    patch_names=patch_names,
    paulis=target_paulis,
    sides=z_normalized_sides,
    interface_paulis=z_interfaces,
    rounds=2,
    coupler_name='route_zzzx',
    route_padding=8,
    route_width=1,  # legacy DEM-validated thin route; not the full ancillary patch diagram
)

for key, value in info.items():
    print(f'{key}: {value}')

cp = system.coupler_patches['route_zzzx']
counts = {}
weights = {}
for stab in cp.stabilizers:
    counts[stab['type']] = counts.get(stab['type'], 0) + 1
    weights[len(stab['pauli'])] = weights.get(len(stab['pauli']), 0) + 1
print(f'stabilizer types: {counts}')
print(f'stabilizer weights: {weights}')
dem = circuit.detector_error_model(decompose_errors=True)
print(f'DEM OK: {dem.num_detectors} det, {dem.num_observables} obs')
print('Legacy thin-route DEM check only; see Exp 2 for the full ancillary-patch detslice diagram.')


---

## Exp 2: native mixed-interface ZZZX full ancillary patch

Here the selected sides have native interface basis `Z, Z, Z, X`, so no logical H is needed. The routed coupler is built with `mixed_stabilizers=True` and `route_width = 2*d - 1`. For full-width routing, data patches must sit on a common coarse grid whose cell size is `route_width`; with the current unrotated parity alignment, the d=3 examples use 10-coordinate data-patch spacing so the intervening 5x5 coarse cells become ancillary patch blocks. The ancillary bus is then built from complete patch-sized cells instead of a thin skeleton path.  For a mixed bus, the ancillary data are prepared in the conjugate basis of the local route label (`Z` route cells prepared in `X`, `X` route cells prepared in `Z`) and read out in the route basis.  That lets the tracker close, gives one mixed-surgery observable, and makes Stim's `detslice-with-ops-svg` render a detector/observable-colored diagram like `multi_patch_LS.ipynb`.


In [ ]:
d = 3
system = make_system(d)
mixed_sides = ['bottom', 'top', 'top', 'left']
native_interfaces = infer_interface_paulis(system, patch_names, mixed_sides)
print(f'native interfaces: {native_interfaces}')
print(f'H indices: {basis_change_indices_for_interfaces(target_paulis, native_interfaces)}')

system.register_coupler(
    UnrotatedRoutedMultiPatchCoupler(), patch_names, 'mixed_geom',
    sides=mixed_sides,
    interface_paulis=native_interfaces,
    target_paulis=list(target_paulis),
    mixed_stabilizers=True,
    route_padding=8,
    route_width=2 * d - 1,
)
cp = system.coupler_patches['mixed_geom']
counts = {}
weights = {}
for stab in cp.stabilizers:
    counts[stab['type']] = counts.get(stab['type'], 0) + 1
    weights[len(stab['pauli'])] = weights.get(len(stab['pauli']), 0) + 1
print(f'mixed-template types: {counts}')
print(f'mixed-template weights: {weights}')
print(f'route_width: {cp.route_width}')
print(f'conflicting patch stabilizers paused: {len(cp.conflicting_stabilizer_coords)}')

from IPython.display import SVG, display

def _coord_of_term(system, term):
    if isinstance(term, tuple):
        return term
    return system.qubit_coords[term]

def _fmt_coord(coord):
    return f'({coord[0]:.0f},{coord[1]:.0f})'

typed_stabilizers = sorted(
    enumerate(cp.stabilizers),
    key=lambda item: (item[1]['syn_coord'][1], item[1]['syn_coord'][0], item[0]),
)
print('\nstabilizer index/type/weight/syn_coord/terms:')
for stab_idx, stab in typed_stabilizers:
    terms = []
    for term, pauli in sorted(stab['pauli'].items(), key=lambda kv: _coord_of_term(system, kv[0])):
        terms.append(f'{pauli}@{_fmt_coord(_coord_of_term(system, term))}')
    print(f'{stab_idx:02d}: {stab["type"]:<5} w={len(stab["pauli"])} '
          f'syn={_fmt_coord(stab["syn_coord"])} terms={", ".join(terms)}')

def render_stabilizer_type_map(system, coupler_name, patch_names, selected_sides=None):
    cp = system.coupler_patches[coupler_name]
    patch_items = [(name, system.patches[name][0]) for name in patch_names]
    selected_side_by_patch = dict(zip(patch_names, selected_sides or []))
    coords = []
    for _, patch in patch_items:
        coords.extend(patch.index_map.keys())
    coords.extend(cp.index_map.keys())
    coords.extend(cp.route_coord_basis.keys())
    coords.extend(stab['syn_coord'] for stab in cp.stabilizers)
    min_x, max_x = min(x for x, _ in coords) - 2, max(x for x, _ in coords) + 2
    min_y, max_y = min(y for _, y in coords) - 2, max(y for _, y in coords) + 2
    scale = 26
    pad = 40
    width = int((max_x - min_x) * scale + 2 * pad)
    height = int((max_y - min_y) * scale + 2 * pad)

    def xy(coord):
        return pad + (coord[0] - min_x) * scale, pad + (coord[1] - min_y) * scale

    pieces = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        '<style>text{font-family:Arial, sans-serif;} .small{font-size:10px;} .label{font-size:11px;font-weight:700;}</style>',
    ]

    for name, patch in patch_items:
        x0, x1, y0, y1 = patch._get_bounds()
        rx0, ry0 = xy((x0 - 0.5, y0 - 0.5))
        rx1, ry1 = xy((x1 + 0.5, y1 + 0.5))
        pieces.append(f'<rect x="{rx0:.1f}" y="{ry0:.1f}" width="{rx1-rx0:.1f}" height="{ry1-ry0:.1f}" fill="none" stroke="#9ca3af" stroke-width="1.4"/>')
        pieces.append(f'<text x="{rx0:.1f}" y="{ry0-6:.1f}" class="label" fill="#374151">{name}</text>')
        edge_specs = {
            'left': ((x0 - 0.5, y0 - 0.5), (x0 - 0.5, y1 + 0.5), (-13, 4)),
            'right': ((x1 + 0.5, y0 - 0.5), (x1 + 0.5, y1 + 0.5), (13, 4)),
            'top': ((x0 - 0.5, y0 - 0.5), (x1 + 0.5, y0 - 0.5), (0, -8)),
            'bottom': ((x0 - 0.5, y1 + 0.5), (x1 + 0.5, y1 + 0.5), (0, 16)),
        }
        for side, (start, end, label_offset) in edge_specs.items():
            basis = UnrotatedRoutedMultiPatchCoupler.infer_side_basis(patch, side)
            color = '#ef4444' if basis == 'X' else '#2563eb'
            sx, sy = xy(start)
            ex, ey = xy(end)
            selected = selected_side_by_patch.get(name) == side
            stroke = 7 if selected else 4
            pieces.append(f'<line x1="{sx:.1f}" y1="{sy:.1f}" x2="{ex:.1f}" y2="{ey:.1f}" stroke="{color}" stroke-width="{stroke}" stroke-linecap="round" opacity="0.95"/>')
            lx, ly = (sx + ex) / 2 + label_offset[0], (sy + ey) / 2 + label_offset[1]
            label = f'{basis}*' if selected else basis
            pieces.append(f'<text x="{lx:.1f}" y="{ly:.1f}" class="label" text-anchor="middle" fill="{color}">{label}</text>')
        for coord in sorted(patch.data_coords):
            x, y = xy(coord)
            pieces.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="2.3" fill="#d1d5db"/>')

    route_colors = {'X': '#fee2e2', 'Z': '#dbeafe'}
    cell_size = scale * 0.88
    for coord, basis in sorted(cp.route_coord_basis.items(), key=lambda kv: (kv[0][1], kv[0][0])):
        x, y = xy(coord)
        pieces.append(f'<rect x="{x-cell_size/2:.1f}" y="{y-cell_size/2:.1f}" width="{cell_size:.1f}" height="{cell_size:.1f}" fill="{route_colors.get(basis, "#f3f4f6")}" stroke="#cbd5e1" stroke-width="0.45"/>')

    for uid in sorted(cp.data_indices):
        x, y = xy(cp.qubit_coords[uid])
        pieces.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="3.0" fill="#111827"/>')
    for uid in sorted(cp.syndrome_indices_x):
        x, y = xy(cp.qubit_coords[uid])
        pieces.append(f'<rect x="{x-3.6:.1f}" y="{y-3.6:.1f}" width="7.2" height="7.2" fill="#ef4444"/>')
    for uid in sorted(cp.syndrome_indices_z):
        x, y = xy(cp.qubit_coords[uid])
        pieces.append(f'<rect x="{x-3.6:.1f}" y="{y-3.6:.1f}" width="7.2" height="7.2" fill="#2563eb"/>')

    typed = sorted(
        enumerate(cp.stabilizers),
        key=lambda item: (item[1]['syn_coord'][1], item[1]['syn_coord'][0], item[0]),
    )
    stab_colors = {'X': '#ef4444', 'Z': '#2563eb', 'MIXED': '#7c3aed'}
    for stab_idx, stab in typed:
        x, y = xy(stab['syn_coord'])
        stype = stab['type']
        short = 'M' if stype == 'MIXED' else stype
        weight = len(stab['pauli'])
        pieces.append(f'<rect x="{x-8:.1f}" y="{y-8:.1f}" width="16" height="16" rx="2" fill="{stab_colors[stype]}" stroke="#111827" stroke-width="0.8"/>')
        pieces.append(f'<text x="{x:.1f}" y="{y+3.6:.1f}" class="label" text-anchor="middle" fill="white">{short}</text>')
        pieces.append(f'<text x="{x:.1f}" y="{y-10.5:.1f}" class="small" text-anchor="middle" fill="#111827">{stab_idx}:w{weight}</text>')

    legend_x, legend_y = 18, 18
    pieces.append(f'<text x="{legend_x}" y="{legend_y}" class="label" fill="#111827">Patch edges: red=X, blue=Z, * = selected bus interface</text>')
    pieces.append(f'<text x="{legend_x}" y="{legend_y + 14}" class="label" fill="#111827">Stabilizer labels: index:w(weight), square color/type</text>')
    for offset, (stype, color) in enumerate(stab_colors.items()):
        y = legend_y + 34 + 18 * offset
        pieces.append(f'<rect x="{legend_x}" y="{y-10}" width="12" height="12" fill="{color}"/>')
        pieces.append(f'<text x="{legend_x+18}" y="{y}" class="small" fill="#111827">{stype}</text>')
    pieces.append('</svg>')
    return ''.join(pieces)

print('\nstatic ancillary patch geometry / stabilizer-type map:')
display(SVG(render_stabilizer_type_map(system, 'mixed_geom', patch_names, mixed_sides)))

prep_basis = routed_coupler_data_basis(system, 'mixed_geom', mode='opposite')
readout_basis = routed_coupler_data_basis(system, 'mixed_geom', mode='same')
zzzx_decomp = solve_routed_pauli_product_syndromes(
    system=system,
    patch_names=patch_names,
    paulis=target_paulis,
    coupler_name='mixed_geom',
    ancilla_readout_bases=readout_basis,
)
print('\nphysical readout-constrained product decomposition:')
print(f'  verified: {zzzx_decomp.verified}')
print(f'  syndrome terms: {len(zzzx_decomp.selected_terms)}')
print(f'  ancilla readout terms: {len(zzzx_decomp.selected_ancilla_terms)}')
for term in zzzx_decomp.selected_ancilla_terms:
    print(f'    {term.pauli}@{term.coord} global_q={term.global_qubit_index}')

tracker = SyndromeTracker(num_qubits=system.num_qubits, expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=True)
builder.write_coordinates()

data_prep = {
    q: 'X'
    for q in system.data_indices
    if system.index_to_owner_map.get(q) != 'mixed_geom'
}
builder.initialize(data_prep, n=system.num_qubits)
builder.apply_syndrome_extraction(
    circuit_chunk=UnrotatedSurfaceCodeExtractionBlock(system).circuit,
    rounds=2,
)

builder.activate_coupler('mixed_geom')
builder.initialize(prep_basis, n=system.num_qubits)
builder.apply_syndrome_extraction(
    circuit_chunk=UnrotatedSurfaceCodeExtractionBlock(system).circuit,
    rounds=2,
)
builder.apply_data_readout({**data_prep, **readout_basis})

circuit = builder.circuit
dem = circuit.detector_error_model(decompose_errors=True)
print('\nfull mixed tracker/observable circuit:')
print(f'  num_qubits: {circuit.num_qubits}')
print(f'  num_detectors: {circuit.num_detectors}')
print(f'  num_observables: {circuit.num_observables}')
print(f'  DEM OK: {dem.num_detectors} det, {dem.num_observables} obs')
print('Stim detslice-with-ops-svg detector/observable diagram:')
display(circuit.diagram('detslice-with-ops-svg'))


---

## Exp 3: full ancillary patch X1Z2 product algebra and diagrams

This is the two-patch native mixed-interface case on the same coarse-grid rule: `p1` connects through an `X` boundary and `p2` connects through a `Z` boundary. The cell solves the physical route-basis readout decomposition for `X1Z2`; depending on the routed geometry, the decomposition may or may not need extra ancilla readout terms. It then shows the same two views as Exp 2: a static ancillary-patch/stabilizer-type map and a full `if_detector=True` Stim `detslice-with-ops-svg` diagram.


In [ ]:
from IPython.display import SVG, display

xz_patch_names = ['p1', 'p2']
xz_sides = ['right', 'top']
xz_target_paulis = 'XZ'

xz_system = QECSystem()
xz_system.add_patch(UnrotatedSurfaceCode(distance=3), name='p1', offset=(0, 0))
xz_system.add_patch(UnrotatedSurfaceCode(distance=3), name='p2', offset=(10, 10))
xz_system.register_coupler(
    UnrotatedRoutedMultiPatchCoupler(), xz_patch_names, 'xz',
    sides=xz_sides,
    interface_paulis=['X', 'Z'],
    target_paulis=list(xz_target_paulis),
    mixed_stabilizers=True,
    route_padding=8,
    route_width=2 * 3 - 1,
)

xz_prep_basis = routed_coupler_data_basis(xz_system, 'xz', mode='opposite')
xz_readout_basis = routed_coupler_data_basis(xz_system, 'xz', mode='same')
xz_decomp = solve_routed_pauli_product_syndromes(
    system=xz_system,
    patch_names=xz_patch_names,
    paulis=xz_target_paulis,
    coupler_name='xz',
    ancilla_readout_bases=xz_readout_basis,
)

print(f'route_width: {xz_system.coupler_patches["xz"].route_width}')
print('interface paulis:', ['X', 'Z'])
print('prep basis counts:', {b: list(xz_prep_basis.values()).count(b) for b in ['X', 'Z']})
print('readout basis counts:', {b: list(xz_readout_basis.values()).count(b) for b in ['X', 'Z']})
print(f'verified: {xz_decomp.verified}')
print(f'syndrome terms: {len(xz_decomp.selected_terms)}')
for term in xz_decomp.selected_terms:
    print(f'  {term.source:<7} {term.stype:<5} syn={term.syn_coord} rec[{term.rec_offset}]')
ancilla_terms = getattr(xz_decomp, 'selected_ancilla_terms', [])
print(f'ancilla readout terms: {len(ancilla_terms)}')
for term in ancilla_terms:
    print(f'  {term.pauli}@{term.coord} global_q={term.global_qubit_index}')

print('\nstatic ancillary patch geometry / stabilizer-type map:')
display(SVG(render_stabilizer_type_map(xz_system, 'xz', xz_patch_names, xz_sides)))

xz_tracker = SyndromeTracker(num_qubits=xz_system.num_qubits, expected_num_logicals=xz_system.num_logicals)
xz_builder = CircuitBuilder(tracker=xz_tracker, system_config=xz_system, if_detector=True)
xz_builder.write_coordinates()

xz_data_prep = {
    q: 'X'
    for q in xz_system.data_indices
    if xz_system.index_to_owner_map.get(q) != 'xz'
}
xz_builder.initialize(xz_data_prep, n=xz_system.num_qubits)
xz_builder.apply_syndrome_extraction(
    circuit_chunk=UnrotatedSurfaceCodeExtractionBlock(xz_system).circuit,
    rounds=2,
)

xz_builder.activate_coupler('xz')
xz_builder.initialize(xz_prep_basis, n=xz_system.num_qubits)
xz_builder.apply_syndrome_extraction(
    circuit_chunk=UnrotatedSurfaceCodeExtractionBlock(xz_system).circuit,
    rounds=2,
)
xz_builder.apply_data_readout({**xz_data_prep, **xz_readout_basis})

xz_circuit = xz_builder.circuit
xz_dem = xz_circuit.detector_error_model(decompose_errors=True)
print('\nfull X1Z2 mixed tracker/observable circuit:')
print(f'  num_qubits: {xz_circuit.num_qubits}')
print(f'  num_detectors: {xz_circuit.num_detectors}')
print(f'  num_observables: {xz_circuit.num_observables}')
print(f'  DEM OK: {xz_dem.num_detectors} det, {xz_dem.num_observables} obs')
print('Stim detslice-with-ops-svg detector/observable diagram:')
display(xz_circuit.diagram('detslice-with-ops-svg'))
